# Etapa 0 — Limpeza e Pré-Processamento

Pipeline idêntico ao REF1 (Sultan et al., 2025):

1. Filtrar amostras `Status != "Results Available"` (remove 79 erros Aspen)
2. Split 80/20 aleatório (`random_state=42`) no conjunto de treinamento
3. Ajustar `MinMaxScaler` **apenas** no treino → aplicar em val e test
4. Salvar arrays `.npy` e CSVs em `processed/`

---

**Mapeamento de colunas (dataset → paper REF1)**

| Dataset | Paper | Faixa | Descrição |
|---------|-------|-------|-----------|
| `Recomp` | P1 | 50–100 bar | Pressão do reator |
| `HX1` | T1 | 200–300 °C | Temperatura do reator |
| `HX2` | T2 | 85–95 °C | Temperatura de entrada na coluna |
| `C1 RR` | RRC1 | 1–10 | Razão de refluxo — coluna 1 |
| `C1 BR` | BRC1 | 0.5–10 | Razão de boil-up — coluna 1 |
| `C2 RR` | RRC2 | 1–10 | Razão de refluxo — coluna 2 |
| `C2 BR` | BRC2 | 0.5–10 | Razão de boil-up — coluna 2 |
| `RFF` | RFF | 0.01–0.25 | **Fração de purga** (≠ papel; não inverter antes do treino) |

> **Quirk RFF:** o dataset representa fração de purga (0.01–0.25); o paper usa fração de reciclo (0.75–0.99). Conversão `RFF_paper = 1 − RFF_dataset` deve ser aplicada **apenas ao reportar resultados do otimizador**.

In [1]:
import pathlib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

HERE         = pathlib.Path(".")
TRAIN_XLSX   = HERE / "Training-Data-Formated.xlsx"
TEST_XLSX    = HERE / "Test-Data-Formated.xlsx"
OUT_DIR      = HERE / "processed"
OUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

# Nomes reais no dataset
INPUTS  = ["Recomp", "HX1", "HX2", "C1 RR", "C1 BR", "C2 RR", "C2 BR", "RFF"]
OUTPUTS = ["Methanol Flow", "Purity", "Energy"]

# Rótulos curtos (usados nos artefatos salvos e nos gráficos)
INPUT_LABELS  = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
OUTPUT_LABELS = ["M_CH3OH", "x_CH3OH", "ET"]

print("Configuração carregada.")

Configuração carregada.


## 1. Carregar e filtrar dados

In [2]:
df_train_raw = pd.read_excel(TRAIN_XLSX)
df_test_raw  = pd.read_excel(TEST_XLSX)

print(f"Training bruto:  {df_train_raw.shape}")
print(f"Test bruto:      {df_test_raw.shape}")
print(f"\nDistribuição de Status (training):")
print(df_train_raw["Status"].value_counts())

# Filtro principal: remover amostras com erros do Aspen
df_train = df_train_raw[df_train_raw["Status"] == "Results Available"].reset_index(drop=True)
df_test  = df_test_raw[df_test_raw["Status"] == "Results Available"].reset_index(drop=True)

print(f"\nApós filtro — Training: {len(df_train)}  |  Test: {len(df_test)}")

X_all      = df_train[INPUTS].values
y_all      = df_train[OUTPUTS].values
X_test_raw = df_test[INPUTS].values
y_test_raw = df_test[OUTPUTS].values

Training bruto:  (2000, 14)
Test bruto:      (200, 14)

Distribuição de Status (training):
Status
Results Available                1921
Results Available with Errors      79
Name: count, dtype: int64

Após filtro — Training: 1921  |  Test: 193


## 2. Verificação de valores ausentes

In [3]:
nan_train = df_train[INPUTS + OUTPUTS].isnull().sum()
nan_test  = df_test[INPUTS + OUTPUTS].isnull().sum()

print("NaN — Training:")
print(nan_train[nan_train > 0] if nan_train.any() else "  Nenhum.")
print("\nNaN — Test:")
print(nan_test[nan_test > 0] if nan_test.any() else "  Nenhum.")

NaN — Training:
  Nenhum.

NaN — Test:
  Nenhum.


## 3. Split treino / validação (80/20)

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Treino:    {X_train.shape[0]} amostras  ({X_train.shape[0]/len(X_all)*100:.1f}%)")
print(f"Validação: {X_val.shape[0]} amostras  ({X_val.shape[0]/len(X_all)*100:.1f}%)")
print(f"Teste:     {X_test_raw.shape[0]} amostras  (fixo, não participa do treino)")

Treino:    1536 amostras  (80.0%)
Validação: 385 amostras  (20.0%)
Teste:     193 amostras  (fixo, não participa do treino)


## 4. Normalização MinMaxScaler

O scaler é ajustado **somente** no conjunto de treino para evitar data leakage.

In [5]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_s = scaler_X.fit_transform(X_train)
y_train_s = scaler_y.fit_transform(y_train)

X_val_s  = scaler_X.transform(X_val)
y_val_s  = scaler_y.transform(y_val)

X_test_s = scaler_X.transform(X_test_raw)
y_test_s = scaler_y.transform(y_test_raw)

# Verificar extrapolação (valores fora de [0,1] indicam test fora do domínio de treino)
for name, arr in [("X_val", X_val_s), ("y_val", y_val_s),
                  ("X_test", X_test_s), ("y_test", y_test_s)]:
    n_oor = int(np.sum((arr < 0) | (arr > 1)))
    status = f"⚠ {n_oor} valor(es) fora de [0,1]" if n_oor else "OK"
    print(f"  {name:8s}: {status}")

print("\nFaixas dos scalers (escala original):")
df_scaler = pd.DataFrame({
    "input": INPUT_LABELS,
    "min":   scaler_X.data_min_,
    "max":   scaler_X.data_min_ + scaler_X.data_range_,
})
print(df_scaler.to_string(index=False))

  X_val   : ⚠ 5 valor(es) fora de [0,1]
  y_val   : ⚠ 1 valor(es) fora de [0,1]
  X_test  : ⚠ 1 valor(es) fora de [0,1]
  y_test  : ⚠ 2 valor(es) fora de [0,1]

Faixas dos scalers (escala original):
input        min        max
   P1  50.027703  99.995530
   T1 200.119416 299.963169
   T2  85.002323  94.995231
 RRC1   1.006756   9.993207
 BRC1   0.502164   9.993719
 RRC2   1.000597   9.998234
 BRC2   0.501839   9.998149
  RFF   0.010007   0.249891


## 5. Salvar artefatos

In [6]:
# Arrays numpy (escala normalizada)
arrays_scaled = {
    "X_train": X_train_s, "y_train": y_train_s,
    "X_val":   X_val_s,   "y_val":   y_val_s,
    "X_test":  X_test_s,  "y_test":  y_test_s,
}
# Arrays numpy (escala original — necessários para o otimizador inverter a normalização)
arrays_raw = {
    "X_train_raw": X_train,     "y_train_raw": y_train,
    "X_val_raw":   X_val,       "y_val_raw":   y_val,
    "X_test_raw":  X_test_raw,  "y_test_raw":  y_test_raw,
}

for name, arr in {**arrays_scaled, **arrays_raw}.items():
    np.save(OUT_DIR / f"{name}.npy", arr)

# CSVs legíveis com cabeçalhos (escala normalizada)
for prefix, X, y in [("train", X_train_s, y_train_s),
                     ("val",   X_val_s,   y_val_s),
                     ("test",  X_test_s,  y_test_s)]:
    pd.DataFrame(X, columns=INPUT_LABELS).to_csv(OUT_DIR / f"X_{prefix}.csv", index=False)
    pd.DataFrame(y, columns=OUTPUT_LABELS).to_csv(OUT_DIR / f"y_{prefix}.csv", index=False)

# Parâmetros dos scalers
np.save(OUT_DIR / "scaler_X_min.npy",   scaler_X.data_min_)
np.save(OUT_DIR / "scaler_X_scale.npy", scaler_X.data_range_)
np.save(OUT_DIR / "scaler_y_min.npy",   scaler_y.data_min_)
np.save(OUT_DIR / "scaler_y_scale.npy", scaler_y.data_range_)

print("Artefatos salvos em", OUT_DIR.resolve())
print(sorted(p.name for p in OUT_DIR.iterdir()))

Artefatos salvos em /Users/lorenzoferreira/Documents/UFRGS/TCC_SBO/ARTEFATOS/ETAPA_0/processed
['X_test.csv', 'X_test.npy', 'X_test_raw.npy', 'X_train.csv', 'X_train.npy', 'X_train_raw.npy', 'X_val.csv', 'X_val.npy', 'X_val_raw.npy', 'scaler_X_min.npy', 'scaler_X_scale.npy', 'scaler_y_min.npy', 'scaler_y_scale.npy', 'y_test.csv', 'y_test.npy', 'y_test_raw.npy', 'y_train.csv', 'y_train.npy', 'y_train_raw.npy', 'y_val.csv', 'y_val.npy', 'y_val_raw.npy']


## 6. Resumo estatístico

In [7]:
print("=== Inputs — escala original (treino) ===")
display(pd.DataFrame(X_train, columns=INPUT_LABELS).describe().loc[["min","mean","max"]].T)

print("\n=== Outputs — escala original (treino) ===")
display(pd.DataFrame(y_train, columns=OUTPUT_LABELS).describe().loc[["min","mean","max"]].T)

print("\n=== Inputs — normalizado (treino) ===")
display(pd.DataFrame(X_train_s, columns=INPUT_LABELS).describe().loc[["min","mean","max"]].T)

print("\n=== Outputs — normalizado (treino) ===")
display(pd.DataFrame(y_train_s, columns=OUTPUT_LABELS).describe().loc[["min","mean","max"]].T)

=== Inputs — escala original (treino) ===


,min,mean,max
P1,50.027703,75.062659,99.995530
T1,200.119416,249.652881,299.963169
T2,85.002323,90.014438,94.995231
RRC1,1.006756,5.510176,9.993207
BRC1,0.502164,5.312467,9.993719
RRC2,1.000597,5.555001,9.998234
BRC2,0.501839,5.288440,9.998149
RFF,0.010007,0.130117,0.249891



=== Outputs — escala original (treino) ===


,min,mean,max
M_CH3OH,375.797579,5705.778206,13151.847678
x_CH3OH,0.638650,0.952058,0.998268
ET,8768.459275,36985.596037,105554.488785



=== Inputs — normalizado (treino) ===


,min,mean,max
P1,0.0,0.501022,1.0
T1,0.0,0.496110,1.0
T2,0.0,0.501567,1.0
RRC1,0.0,0.501134,1.0
BRC1,0.0,0.506798,1.0
RRC2,0.0,0.506178,1.0
BRC2,0.0,0.504049,1.0
RFF,0.0,0.500701,1.0



=== Outputs — normalizado (treino) ===


,min,mean,max
M_CH3OH,0.0,0.417185,1.0
x_CH3OH,0.0,0.871503,1.0
ET,0.0,0.291541,1.0
